# Metaorder signal — deep tuning report (Binance BTCUSDT)

This notebook is meant to read like a **mini research memo**: timeline of data splits, a **128-point** validation grid (`fine` preset), a **leaderboard** of top validation configs evaluated on **held-out test** (overfitting check), heatmaps, distributions, and **benchmarked** equity vs buy-and-hold on the same tape.

**Methodology:** structural parameters are fit **only** on **calibration**. Signal and execution hyperparameters are grid-searched on **validation** (maximize validation final equity). The winner is run **once** on **test**.

**Caveat:** Strong visuals do not imply live profitability — costs are stylised and the search still risks **multiple-testing** bias even with a clean test segment.

In [ ]:
%matplotlib inline
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
if not (REPO / "src").is_dir():
    REPO = REPO.parent
os.chdir(REPO)
print("repo root:", REPO.resolve())

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["figure.dpi"] = 120
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    try:
        plt.style.use("ggplot")
    except OSError:
        pass

In [ ]:
import os

from metaorder_signal.empirical.binance_public import fetch_agg_trades
from metaorder_signal.empirical.calibration import calibrate_from_trades
from metaorder_signal.empirical.tuning import (
    grid_preset_size,
    grid_search_signal_backtest,
    three_way_time_split,
)

CACHE = REPO / "data" / "binance_BTCUSDT_tuning.csv"
# More trades → smoother curves & stabler splits (delete cache or set METAORDER_REFRESH_CACHE=1 to refetch)
MAX_TRADES = 36_000
REFRESH = os.environ.get("METAORDER_REFRESH_CACHE", "").lower() in ("1", "true", "yes")

CACHE.parent.mkdir(parents=True, exist_ok=True)
if CACHE.is_file() and not REFRESH:
    # Avoid parse_dates= — CSV rows can mix subsecond precision; one strict format then breaks.
    tape = pd.read_csv(CACHE)
    tape["timestamp"] = pd.to_datetime(tape["timestamp"], utc=True, format="mixed")
    print(f"Loaded cache {CACHE} rows={len(tape):,}")
else:
    if CACHE.is_file() and REFRESH:
        print("Refreshing cache from Binance…")
    else:
        print("Fetching Binance (requires network)…")
    raw = fetch_agg_trades("BTCUSDT", max_trades=MAX_TRADES)
    tape = raw[["timestamp", "mid", "quantity", "sign"]].copy()
    tape.to_csv(CACHE, index=False)
    print(f"Saved {len(tape):,} rows to {CACHE}")

tape.head()

In [ ]:
cal, val, tst = three_way_time_split(tape, calib_frac=0.40, val_frac=0.30)
print("calib", len(cal), "val", len(val), "test", len(tst))
for name, seg in (("calib", cal), ("val", val), ("test", tst)):
    print(
        f"  {name}: {seg['timestamp'].iloc[0]} → {seg['timestamp'].iloc[-1]}"
    )

cp = calibrate_from_trades(cal, min_runs=40)
structural = cp.to_signal_params()
print("alpha=", structural.alpha, "sigma_d=", structural.sigma_d, "vd=", structural.vd)

In [ ]:
PRESET = "fine"
n_grid = grid_preset_size(PRESET)
print(f"Grid search on VALIDATION only ({n_grid} configs, preset={PRESET})…")
result = grid_search_signal_backtest(
    structural,
    val,
    tst,
    s_max=8000.0,
    maximize="final_equity",
    grid_preset=PRESET,
    leaderboard_k=10,
)
print("Best config:\n", json.dumps(result.best_config, indent=2))
print("Validation metrics:", json.dumps(result.validation_metrics, indent=2))
print("TEST metrics:", json.dumps(result.test_metrics, indent=2))
print("\nLeaderboard — top validation configs scored on held-out test:")
print(pd.DataFrame(result.leaderboard).to_string(index=False))

### Heatmaps (`p_min` × `phi_entry`)

The **fine** preset uses **4×4** levels here (was 3×2 in the quick grid), so you should see a **real surface**, not three chunky stripes.

**Left:** **Max** validation final equity over all other tuned knobs per `(p_min, phi_entry)` cell — an **upper envelope** (how good validation can get if everything else cooperates).

**Right:** Same pairs with **`rho_max`, `n_min`, spread, `kappa` frozen** to the printed `best_config` — **marginal** sensitivity of these two knobs only.

Numbers are printed in each cell so tiny colour differences remain visible.

In [ ]:
from metaorder_signal.backtest import BacktestConfig, run_event_backtest
from metaorder_signal.signal import SignalConfig

g = result.grid_rows.copy()
pivot = g.pivot_table(
    index="p_min",
    columns="phi_entry",
    values="val_final_equity",
    aggfunc="max",
)

bcfg_fix = BacktestConfig(half_spread_frac=result.best_config["half_spread"])
fixed_rows = []
for pm in pivot.index:
    for pe in pivot.columns:
        scfg = SignalConfig(
            p_min=float(pm),
            phi_entry=float(pe),
            rho_max=result.best_config["rho_max"],
            n_min=int(result.best_config["n_min"]),
            phi_exit=result.best_config["phi_exit"],
            s_max=result.best_config["s_max"],
            kappa_cost=result.best_config["kappa"],
        )
        out = run_event_backtest(val, structural, scfg, bcfg_fix)
        fixed_rows.append(
            {"p_min": pm, "phi_entry": pe, "eq": float(out["equity"].iloc[-1])}
        )
gf = pd.DataFrame(fixed_rows)
pivot_fix = gf.pivot(index="p_min", columns="phi_entry", values="eq")

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11.5, 5))

for ax, pv, title in (
    (ax0, pivot, "Max val equity (re-tune other knobs per cell)"),
    (ax1, pivot_fix, "Val equity (other knobs = best_config)"),
):
    arr = pv.values.astype(float)
    im = ax.imshow(arr, aspect="auto", cmap="RdYlGn")
    ax.set_xticks(range(len(pv.columns)))
    ax.set_xticklabels([f"{x:.2f}" for x in pv.columns])
    ax.set_yticks(range(len(pv.index)))
    ax.set_yticklabels([f"{x:.2f}" for x in pv.index])
    ax.set_xlabel("phi_entry")
    ax.set_ylabel("p_min")
    ax.set_title(title)
    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            ax.text(j, i, f"{arr[i, j]:.4f}", ha="center", va="center", color="black", fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(REPO / "notebooks" / "fig_heatmap_val_equity.png", dpi=160)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(g["val_final_equity"], bins=32, color="steelblue", edgecolor="white", alpha=0.88)
axes[0].axvline(0, color="black", lw=1)
axes[0].axvline(result.best_validation_equity, color="crimson", lw=2, label="best val")
axes[0].legend(fontsize=8)
axes[0].set_title("Validation final equity — full grid")
axes[0].set_xlabel("final_equity")

axes[1].hist(g["val_sharpe_like"].replace([np.inf, -np.inf], np.nan).dropna(), bins=20, color="seagreen", edgecolor="white")
axes[1].set_title("Validation Sharpe-like (equity steps)")
axes[1].set_xlabel("Sharpe-like")

plt.tight_layout()
plt.savefig(REPO / "notebooks" / "fig_grid_distributions.png", dpi=140)
plt.show()

### Leaderboard — does validation ranking survive the test segment?

Each bar pair is a **distinct** hyperparameter set ranked by validation equity. **Orange** is the same strategy run forward on **test** without any refitting — gaps between blue and orange at rank 1 are the classical **overfitting** signature.

In [ ]:
lb = pd.DataFrame(result.leaderboard)
fig, ax = plt.subplots(figsize=(11, 4.5))
x = np.arange(len(lb))
w = 0.36
ax.bar(x - w / 2, lb["val_final_equity"], width=w, label="validation final equity", color="#34495e")
ax.bar(x + w / 2, lb["test_final_equity"], width=w, label="test final equity", color="#e67e22")
ax.set_xticks(x)
ax.set_xticklabels([f"rank {r}" for r in lb["rank"]])
ax.axhline(0, color="black", lw=0.9)
ax.legend()
ax.set_title("Top validation configs — held-out test comparison")
plt.tight_layout()
plt.savefig(REPO / "notebooks" / "fig_leaderboard_val_test.png", dpi=160)
plt.show()

### Chronological coverage

Tape partitioned by **time quantiles** so segments stay causally ordered (no random mixing).

In [ ]:
import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(11, 2.2))
for seg, color, lab in (
    (cal, "#3498db", "calibration (fit structure)"),
    (val, "#2ecc71", "validation (tune grid)"),
    (tst, "#e67e22", "test (score once)"),
):
    t0, t1 = seg["timestamp"].iloc[0], seg["timestamp"].iloc[-1]
    ax.axvspan(t0, t1, alpha=0.35, color=color, label=lab)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
ax.set_yticks([])
ax.set_title("Binance BTCUSDT agg trades — segment timeline")
ax.legend(loc="upper center", ncol=3, fontsize=8, frameon=True)
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig(REPO / "notebooks" / "fig_split_timeline.png", dpi=160)
plt.show()

### Strategy equity vs BTC buy-and-hold, plus underwater drawdown

**Left column:** validation segment — strategy cumulative cash (green axis) vs simple BTC **buy-and-hold return %** from the first tick (blue axis). **Right column:** same on **test**.

Bottom row: normalised **drawdown** of the strategy equity path (relative to running peak). Compare leaderboard bars above — if test strategy sits near zero while BTC moved, the signal may add little beyond timing noise.

In [ ]:
from metaorder_signal.empirical.benchmarks import (
    buy_and_hold_cumulative_return,
    equity_drawdown_series,
)

bcfg = BacktestConfig(half_spread_frac=result.best_config["half_spread"])
scfg = SignalConfig(
    p_min=result.best_config["p_min"],
    phi_entry=result.best_config["phi_entry"],
    rho_max=result.best_config["rho_max"],
    n_min=int(result.best_config["n_min"]),
    phi_exit=result.best_config["phi_exit"],
    s_max=result.best_config["s_max"],
    kappa_cost=result.best_config["kappa"],
)

out_val = run_event_backtest(val, structural, scfg, bcfg)
out_test = run_event_backtest(tst, structural, scfg, bcfg)

strat_v = out_val["equity"].to_numpy(dtype=float)
strat_t = out_test["equity"].to_numpy(dtype=float)
bh_v = buy_and_hold_cumulative_return(val["mid"])
bh_t = buy_and_hold_cumulative_return(tst["mid"])

fig, axes = plt.subplots(2, 2, figsize=(12, 7), gridspec_kw={"height_ratios": [1.15, 0.78]})

for ax_top, strat, bh, title in (
    (axes[0, 0], strat_v, bh_v, "Validation — strategy vs BTC buy&hold"),
    (axes[0, 1], strat_t, bh_t, "Test (held-out)"),
):
    xv = np.arange(len(strat))
    ax_top.plot(xv, strat, color="#27ae60", lw=1.35, label="strategy cash proxy")
    ax2 = ax_top.twinx()
    ax2.plot(xv, bh * 100.0, color="#2980b9", lw=1.05, alpha=0.88, label="BTC B&H %")
    ax_top.axhline(0, color="gray", lw=0.65)
    ax_top.set_title(title)
    ax_top.set_xlabel("tick index (segment-local)")
    ax_top.set_ylabel("strategy equity", color="#27ae60")
    ax2.set_ylabel("BTC total return (%)", color="#2980b9")

dd_v = equity_drawdown_series(strat_v)
dd_t = equity_drawdown_series(strat_t)
axes[1, 0].fill_between(np.arange(len(dd_v)), dd_v, color="#c0392b", alpha=0.42)
axes[1, 0].set_title("Strategy underwater (validation)")
axes[1, 0].set_xlabel("tick index")
axes[1, 0].set_ylabel("drawdown / |peak|")

axes[1, 1].fill_between(np.arange(len(dd_t)), dd_t, color="#c0392b", alpha=0.42)
axes[1, 1].set_title("Strategy underwater (test)")
axes[1, 1].set_xlabel("tick index")

plt.tight_layout()
plt.savefig(REPO / "notebooks" / "fig_equity_benchmark_dd.png", dpi=165)
plt.show()

fig2, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(strat_v, label="validation", color="#2980b9", lw=1.2)
ax.plot(strat_t, label="test (held-out)", color="#e67e22", lw=1.2)
ax.axhline(0, color="gray", lw=0.75)
ax.legend()
ax.set_title("Tuned strategy — validation vs test equity")
ax.set_xlabel("tick index (segment-local)")
ax.set_ylabel("cumulative cash proxy")
plt.tight_layout()
plt.savefig(REPO / "notebooks" / "fig_equity_val_vs_test.png", dpi=160)
plt.show()

print("TEST final equity (strategy):", float(strat_t[-1]))
print("TEST segment BTC buy&hold return:", float(bh_t[-1]))

### Execution stress — wider spreads on **test**

The grid tuned **half_spread_frac**, but live spreads jump around. Here we scale the **winner’s** spread by simple multipliers and **re-run only on test** (same structural params + signal knobs). If bars plunge before **2×**, the edge is **fragile** to friction.

In [ ]:
mults = np.array([1.0, 1.25, 1.5, 2.0, 2.5])
base_spread = float(result.best_config["half_spread"])
stress_rows = []
for m in mults:
    bcfg_s = BacktestConfig(half_spread_frac=base_spread * float(m))
    o_st = run_event_backtest(tst, structural, scfg, bcfg_s)
    stress_rows.append(
        {"spread_mult": m, "test_final_equity": float(o_st["equity"].iloc[-1])}
    )
stress_df = pd.DataFrame(stress_rows)
display(stress_df)

fig, ax = plt.subplots(figsize=(9, 4))
labels = [f"×{m:g}" for m in stress_df["spread_mult"]]
ax.bar(labels, stress_df["test_final_equity"], color="#8e44ad", edgecolor="white", alpha=0.92)
ax.axhline(0, color="black", lw=0.95)
ax.set_xlabel("half_spread multiplier (relative to tuned winner)")
ax.set_ylabel("TEST final equity")
ax.set_title("Winner strategy — spread stress on held-out test")
plt.tight_layout()
plt.savefig(REPO / "notebooks" / "fig_cost_stress_test.png", dpi=165)
plt.show()

### Reproducibility bundle

Writes **`notebooks/tuning_summary.json`** — best config, metrics, leaderboard, and spread-stress table — so you can diff runs without diffing the executed notebook binary.

In [ ]:
summary = {
    "grid_preset": PRESET,
    "n_tape_rows": len(tape),
    "best_config": result.best_config,
    "validation_metrics": result.validation_metrics,
    "test_metrics": result.test_metrics,
    "leaderboard": result.leaderboard,
    "spread_stress_on_test": stress_df.to_dict(orient="records"),
}
out_json = REPO / "notebooks" / "tuning_summary.json"
with open(out_json, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, default=str)
print("Wrote", out_json.resolve())

exec_rows = [
    {
        "segment": "validation",
        "final_equity": result.validation_metrics.get("final_equity"),
        "sharpe_like": result.validation_metrics.get("sharpe_like"),
        "max_dd": result.validation_metrics.get("max_drawdown"),
    },
    {
        "segment": "test",
        "final_equity": result.test_metrics.get("final_equity"),
        "sharpe_like": result.test_metrics.get("sharpe_like"),
        "max_dd": result.test_metrics.get("max_drawdown"),
    },
]
display(pd.DataFrame(exec_rows))

## Figures saved (`notebooks/`)

| Figure | Role |
|--------|------|
| `fig_heatmap_val_equity.png` | 4×4 × dual-panel sensitivity |
| `fig_grid_distributions.png` | Grid score distributions |
| `fig_leaderboard_val_test.png` | Top validation configs vs test |
| `fig_split_timeline.png` | Calibration / val / test timeline |
| `fig_equity_benchmark_dd.png` | Benchmark + drawdown panel |
| `fig_equity_val_vs_test.png` | Strategy equity overlay |
| `fig_cost_stress_test.png` | Spread multipliers on TEST (winner) |

Also **`tuning_summary.json`** — serialised metrics + leaderboard + stress table (small, diff-friendly).

If **test** underperforms validation while BTC drift was favourable, interpret cautiously (**overfitting**, regime shift, or mis-scaled costs).

### Embedded figures (saved alongside this notebook)

![Heatmaps](fig_heatmap_val_equity.png)

![Grid distributions](fig_grid_distributions.png)

![Leaderboard val vs test](fig_leaderboard_val_test.png)

![Split timeline](fig_split_timeline.png)

![Equity benchmark drawdown](fig_equity_benchmark_dd.png)

![Validation vs test equity](fig_equity_val_vs_test.png)

![Cost stress on test](fig_cost_stress_test.png)
